# IMPORTS

In [5]:
from qiskit import *
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_state_city
import qiskit.quantum_info as qi
from qiskit.visualization import plot_histogram
from math import gcd , ceil
from numpy.random import randint
import pandas as pd
from fractions import Fraction
from tqdm import tqdm
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
import matplotlib.pyplot as plt
import random
import sys
import math
import array
import fractions
import numpy as np

In [24]:
from qiskit.circuit.library import QFT, UnitaryGate
from tqdm import tqdm

# Manually calculate period

In [25]:
def period(a,N): # returns r
    new = a
    i = 1
    while new != 1 and i < N:
        new = (new * a) % N
        i+=1
    return i

In [26]:
def modinv(a,N): # returns a^(-1)
    r = period(a,N)
    return (a ** (r-1)) % N

# Defining the oracle

In [27]:
def f(x1,x2,a,b,N):
    out = ((b ** x1) * (a**x2)) % N
    return out


In [59]:
def DLP_oracle(a , b , N , qubits_state , qubits_eval): # Manually constructs Unitary using Qiksit API call
    n_qubits = 2*qubits_eval + qubits_state
    outputs = np.zeros(2 ** n_qubits, dtype=int)
    for state in tqdm(range(2**(n_qubits))):
        state_bin = format(state, f"0{n_qubits}b")
        y_bin = state_bin[:qubits_state]
        x2_bin = state_bin[qubits_state: qubits_state + qubits_eval]
        x1_bin = state_bin[qubits_state + qubits_eval:]
        x1 = int(x1_bin, 2)
        x2 = int(x2_bin, 2)
        y = int(y_bin, 2)
        y_xor_f_bin = format(f(x1, x2,a,b,N) ^ y, f"0{qubits_state}b")
        output_bin = y_xor_f_bin + x2_bin + x1_bin
        outputs[state] = int(output_bin, 2)

    U = np.zeros((2**(n_qubits),2**(n_qubits)),dtype = int)
    for y in tqdm(range(2 ** n_qubits)):
            for i in range(2 ** n_qubits):
                U[i][y] = 1 if outputs[y] == i else 0

    
    return UnitaryGate(U, label=f'Oracle for {a}^x1 * {b}^x2 mod {N}')


# Making Circuit for DLP

In [60]:
def DLP(a,b,N):
    r = period(a,N)
    e_qb = ceil(np.log2(r))
    s_qb = ceil(np.log2(N))
    
    qc = QuantumCircuit(QuantumRegister(e_qb, 'x1'), QuantumRegister(e_qb, 'x2'), QuantumRegister(s_qb, 'y'), ClassicalRegister(e_qb, 'cl_x1'), ClassicalRegister(e_qb, 'cl_x2'))
    for i in range(2*e_qb):
        qc.h(i)
    qc.barrier()

    # Oracle
    oracle = DLP_oracle(a,b,N,s_qb,e_qb)
    qc.append(oracle,range(2*e_qb+s_qb))
    # Inv qft
    qc.barrier()
    inv_QFT = QFT(e_qb,inverse = True, name = 'QFT(-1)').to_gate()
    qc.append(inv_QFT,range(e_qb))
    qc.append(inv_QFT,range(e_qb,2*e_qb))

    print(qc)
    return qc , r , e_qb

    

In [61]:
def eval_DLP(qc,a,b,N,r,e_qb):
    qc.measure(range(e_qb*2),range(e_qb*2))
    backend = AerSimulator()
    n_shots = 32
    
    t_qc = transpile(qc, backend)
    job = backend.run(t_qc, shots=n_shots, memory=True)
    result = job.result()
    memory = result.get_memory()

    for readout in memory:
        # Qiskit bitstrings are right-to-left. 
        x2_str = readout[:e_qb]
        x1_str = readout[e_qb:]

        phase2 = int(x2_str, 2) / (2 ** e_qb)
        phase1 = int(x1_str, 2) / (2 ** e_qb)

        f2 = Fraction(phase2).limit_denominator(r)
        f1 = Fraction(phase1).limit_denominator(r)

        try:
            s2_inv = pow(f2.numerator, -1, r)
            s_candidate = (f1.numerator * s2_inv) % r
            if pow(a, s_candidate, N) == b % N:
                return s_candidate
        except ValueError:
            continue

    return None # If no candidate was found in the shots

In [64]:
a = 2
b = 12
N = 23
qc , r , eqb  = DLP(a,b,N)
s = eval_DLP(qc,a,b,N,r,eqb)

100%|█████████████████████████████████████████████████████████████████████████████| 8192/8192 [00:44<00:00, 184.70it/s]


         ┌───┐ ░ ┌──────────────────────────────────┐ ░ ┌──────────┐
   x1_0: ┤ H ├─░─┤0                                 ├─░─┤0         ├
         ├───┤ ░ │                                  │ ░ │          │
   x1_1: ┤ H ├─░─┤1                                 ├─░─┤1         ├
         ├───┤ ░ │                                  │ ░ │  QFT(-1) │
   x1_2: ┤ H ├─░─┤2                                 ├─░─┤2         ├
         ├───┤ ░ │                                  │ ░ │          │
   x1_3: ┤ H ├─░─┤3                                 ├─░─┤3         ├
         ├───┤ ░ │                                  │ ░ ├──────────┤
   x2_0: ┤ H ├─░─┤4                                 ├─░─┤0         ├
         ├───┤ ░ │                                  │ ░ │          │
   x2_1: ┤ H ├─░─┤5                                 ├─░─┤1         ├
         ├───┤ ░ │                                  │ ░ │  QFT(-1) │
   x2_2: ┤ H ├─░─┤6  Oracle for 2^x1 * 12^x2 mod 23 ├─░─┤2         ├
         ├───┤ ░ │                

In [65]:
print(s)

10
